<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 1b — Modèles de référence : k-mer + ML classique, et one-hot + CNN
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Représenter une séquence d'ADN par des fréquences de k-mers (k=4) et entraîner une régression logistique<br>
    - Encoder les nucléotides en one-hot et entraîner un petit CNN 1D qui apprend ses propres motifs<br>
    - Comparer les deux références sur accuracy, F1, nombre de paramètres et latence<br>
</div>

<div style="border-left: 4px solid #0969da; background-color: #f2f6fd; padding: 0.9em 1.1em; color: #24292f;">
  <strong style="color: #0969da;">Version corrigée &mdash; solution de référence</strong><br>
  Ce notebook contient <strong>toutes les réponses</strong>. Il n'est pas à compléter et n'est pas à rendre.<br>
  Travaillez d'abord sur la version étudiante du dossier parent, et ne consultez ce fichier qu'après avoir essayé par vous-même.
</div>

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

Deux façons de transformer une chaîne d'ADN en quelque chose qu'un modèle peut utiliser,
avant même de toucher à un modèle de langage pré-entraîné :

1. **Fréquences de k-mers** : compter la fréquence de chaque sous-chaîne de longueur k
   (par ex. les 256 4-mers possibles), normaliser -> un vecteur de taille fixe -> ML
   classique (régression logistique).
2. **One-hot + CNN** : encoder chaque nucléotide en un vecteur one-hot de dimension 4
   (A/C/G/T), empiler en un tenseur (4, 200), et laisser un petit CNN 1D apprendre
   lui-même ses propres caractéristiques directement à partir de la séquence brute.


<img src="https://raw.githubusercontent.com/Genereux-akotenou/EEIA-bioAI-Workshop-project/main/day1/assets/illustration2.png"/>

In [3]:
import sys
sys.path.append("src")

import numpy as np
import torch
from data import load_all # importer depuis le dossier src/data.py

splits = load_all("../../2-data/processed")
train, val = splits["train"], splits["val"]

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Modèle de référence 1 : fréquences de k-mers + régression logistique**

**À faire :** 
- complétez le code ci-dessous pour calculer les matrices de k-mers (k=4) sur
train/val, 
- entraîner un classifieur logistique (`make_kmer_classifier("logreg")`), 
- puis l'évaluer sur validation avec `evaluate_sklearn`.

In [4]:
from featurize import kmer_matrix  # importer depuis le dossier src/featurize.py

K = 4
X_train_kmer = kmer_matrix(train["sequence"].tolist(), k=K)
X_val_kmer = kmer_matrix(val["sequence"].tolist(), k=K)
y_train, y_val = train["label"].to_numpy(), val["label"].to_numpy()

In [5]:
from models.baselines import make_kmer_classifier  # importer depuis le dossier src/models

kmer_clf = make_kmer_classifier("logreg")
kmer_clf.fit(X_train_kmer, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [6]:
from eval import evaluate_sklearn # importer depuis le dossier src/eval.py

kmer_metrics = evaluate_sklearn(kmer_clf, X_val_kmer, y_val)
print("k-mer + logreg:", kmer_metrics)

k-mer + logreg: {'accuracy': 0.8481781376518218, 'f1': 0.854718199428097}


#### **Modèle de référence 2 : nucléotides one-hot + petit CNN**

**À faire :** encodez les fenêtres en one-hot (`one_hot_batch`, longueur 200), instanciez
`OneHotCNN`, puis entraînez-le pendant 5 époques (Adam, lr=1e-3,
`binary_cross_entropy_with_logits`, mini-lots de 256).

In [7]:
from featurize import one_hot_batch  # importer depuis le dossier src/featurize.py
from models.baselines import OneHotCNN        # importer depuis le dossier src/models
from eval import evaluate_logits, count_params, measure_latency_sklearn, measure_latency_torch # importer depuis le dossier src/eval.py

In [8]:
WINDOW = 200
X_train_oh = torch.tensor(one_hot_batch(train["sequence"].tolist(), length=WINDOW))
X_val_oh = torch.tensor(one_hot_batch(val["sequence"].tolist(), length=WINDOW))
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

cnn = OneHotCNN(seq_len=WINDOW)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)

n = X_train_oh.shape[0]
for epoch in range(5):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        logits = cnn(X_train_oh[idx])
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_train_t[idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    print(f"epoch {epoch+1}: loss={epoch_loss/n:.4f}")

epoch 1: loss=0.5886
epoch 2: loss=0.4908
epoch 3: loss=0.4075
epoch 4: loss=0.3746
epoch 5: loss=0.3562


In [9]:
with torch.no_grad():
    val_logits = cnn(X_val_oh).numpy()
cnn_metrics = evaluate_logits(val_logits, y_val)
print("one-hot CNN:", cnn_metrics)

one-hot CNN: {'accuracy': 0.8250433776749566, 'f1': 0.8429794964962367}


#### **Comparaison**

In [10]:
import pandas as pd

comparison = pd.DataFrame([
    {"model": "kmer+logreg", **kmer_metrics,
     "params": X_train_kmer.shape[1] + 1,
     "latency_ms": measure_latency_sklearn(kmer_clf, X_val_kmer[:1])},
    {"model": "onehot+CNN", **cnn_metrics,
     "params": count_params(cnn),
     "latency_ms": measure_latency_torch(cnn, X_val_oh[:1])},
])
comparison

,model,accuracy,f1,params,latency_ms
0,kmer+logreg,0.848178,0.854718,257,0.081270
1,onehot+CNN,0.825043,0.842979,10465,0.181191


#### **Point de contrôle**

Vous devriez avoir un tableau accuracy/F1/params/latence pour les deux modèles de
référence. Gardez ces chiffres (ou le DataFrame `comparison`) — ils serviront pour le
graphique d'efficacité du Jour 4.

Suite : `02_evo2_embeddings_and_classifier.ipynb` — un modèle de fondation génomique
fait-il mieux ?

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 1b</div>
      <div>Prochaine &eacute;tape &rarr; <code>day2/02_evo2_embeddings_and_classifier.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>